# EXP-02 — Baseline Algorithms (Window, DBSCAN, HDBSCAN Global)

Development-only baseline evaluation on DAT-02 pilot feature data. Fits three baselines per-mainshock window on 4-D normalized relative spacetime vectors `[x_scaled, y_scaled, z_scaled, time_scaled]`. Evaluates whether density clustering reduces spatial background events relative to a temporal window rule. Run via Restart Kernel → Run All.

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'notebooks/baseline_models.py').is_file(), 'Mở notebook từ root repo hoặc notebooks/'
sys.path.insert(0, str(root / 'notebooks'))
sys.path.insert(0, str(root / 'tests'))

from experiment_harness import load_feature_bundle, MODE_DEVELOPMENT
from baseline_models import (
    run_baseline_experiment,
    generate_local_smoke_report,
    VECTOR_COLUMNS,
)
print('EXP-02 baseline models imported successfully.')

## 1. Load Development Pilot Bundle

Loads the DAT-02 pilot feature bundle. Validates checksums, schema, and finite constraints before fitting.

In [ ]:
from test_experiment_harness import create_pilot_fixture

bundle_path = os.environ.get('PILOT_BUNDLE', '')
_temp_dir = None
if bundle_path and Path(bundle_path).is_dir():
    pilot_dir = Path(bundle_path).resolve()
else:
    # Fall back to self-contained synthetic pilot fixture
    _temp_dir = tempfile.TemporaryDirectory()
    pilot_dir = Path(_temp_dir.name)
    create_pilot_fixture(pilot_dir)

bundle = load_feature_bundle(pilot_dir, mode=MODE_DEVELOPMENT)
assert bundle['mode'] == MODE_DEVELOPMENT
assert bundle['is_official'] is False
assert bundle['manifest']['allowed_usage'] == 'DEVELOPMENT_ONLY'

features = bundle['features_table']
print('Loaded Pilot ID:', bundle['bundle_id'])
print('Feature candidate count:', features.num_rows)
print('4-D Vector columns:', VECTOR_COLUMNS)
assert features.num_rows > 0

## 2. Fit Window Baseline

Applies the pure temporal window counting rule per mainshock window. All POST events in the window are marked as candidate sequence members. Cluster IDs and probabilities are strictly NULL.

In [ ]:
output_env = os.environ.get('EXP02_OUTPUT_DIR')
if output_env:
    exp_out_dir = Path(output_env).resolve()
else:
    exp_out_dir = (root / 'staging/ml-experiments/exp02-baselines').resolve()
exp_out_dir.mkdir(parents=True, exist_ok=True)

cfg_win_path = root / 'config/ml/models/window_baseline_v1.json'
cfg_win = json.loads(cfg_win_path.read_text(encoding='utf-8'))

res_window = run_baseline_experiment(
    bundle=bundle,
    algorithm_name='WINDOW',
    config=cfg_win,
    model_config_version=cfg_win['model_config_version'],
    output_dir=exp_out_dir / 'window',
    mode=MODE_DEVELOPMENT,
    allow_overwrite=True,
)

print('WINDOW Baseline Completed:')
print('  Total candidates processed:', res_window['metrics']['total_candidates'])
print('  Windows processed:', res_window['metrics']['windows_processed'])
print('  Sequence members:', res_window['metrics']['total_sequence_members'])
print('  Aftershocks:', res_window['metrics']['total_aftershocks'])
assert not res_window['write_result']['has_success_marker'], 'Never emit _SUCCESS.json in pilot'

## 3. Fit DBSCAN Global Baseline

Fits DBSCAN per-mainshock window on 4-D normalized relative vectors. Isolates the cluster containing the mainshock as the sequence. Noise points receive label -1; probabilities remain strictly NULL.

In [ ]:
cfg_db_path = root / 'config/ml/models/dbscan_global_dev_v1.json'
cfg_db = json.loads(cfg_db_path.read_text(encoding='utf-8'))

res_dbscan = run_baseline_experiment(
    bundle=bundle,
    algorithm_name='DBSCAN',
    config=cfg_db,
    model_config_version=cfg_db['model_config_version'],
    output_dir=exp_out_dir / 'dbscan',
    mode=MODE_DEVELOPMENT,
    allow_overwrite=True,
)

print('DBSCAN Global Baseline Completed:')
print('  Total candidates processed:', res_dbscan['metrics']['total_candidates'])
print('  Windows processed:', res_dbscan['metrics']['windows_processed'])
print('  Sequence members:', res_dbscan['metrics']['total_sequence_members'])
print('  Noise events detected:', res_dbscan['metrics']['total_noise_candidates'])
assert not res_dbscan['write_result']['has_success_marker']

## 4. Fit HDBSCAN Global Baseline

Fits HDBSCAN per-mainshock window with variable density clustering. Computes calibrated membership probabilities in [0.0, 1.0]. Isolates the cluster containing the mainshock.

In [ ]:
cfg_hdb_path = root / 'config/ml/models/hdbscan_global_dev_v1.json'
cfg_hdb = json.loads(cfg_hdb_path.read_text(encoding='utf-8'))

res_hdbscan = run_baseline_experiment(
    bundle=bundle,
    algorithm_name='HDBSCAN_GLOBAL',
    config=cfg_hdb,
    model_config_version=cfg_hdb['model_config_version'],
    output_dir=exp_out_dir / 'hdbscan_global',
    mode=MODE_DEVELOPMENT,
    allow_overwrite=True,
)

print('HDBSCAN Global Baseline Completed:')
print('  Total candidates processed:', res_hdbscan['metrics']['total_candidates'])
print('  Windows processed:', res_hdbscan['metrics']['windows_processed'])
print('  Sequence members:', res_hdbscan['metrics']['total_sequence_members'])
print('  Noise events detected:', res_hdbscan['metrics']['total_noise_candidates'])
assert not res_hdbscan['write_result']['has_success_marker']

## 5. Cross-Algorithm Reconciliation & Input Immutability

Validates that: (1) all algorithms processed identical candidate counts, (2) density clustering separates background noise, and (3) input candidate table was not mutated.

In [ ]:
total_cands = features.num_rows
assert res_window['metrics']['total_candidates'] == total_cands
assert res_dbscan['metrics']['total_candidates'] == total_cands
assert res_hdbscan['metrics']['total_candidates'] == total_cands

print(f"Reconciliation Table (Total Candidates = {total_cands}):")
print("--------------------------------------------------------------------------------")
print(f"{'Algorithm':<18} | {'Seq Members':<12} | {'Aftershocks':<12} | {'Noise':<8} | {'Status':<10}")
print("--------------------------------------------------------------------------------")
for algo, res in [('WINDOW', res_window), ('DBSCAN', res_dbscan), ('HDBSCAN_GLOBAL', res_hdbscan)]:
    m = res['metrics']
    noise_str = str(m.get('total_noise_candidates', 'N/A'))
    status_str = f"{m['successful_windows']}/{m['windows_processed']} PASS"
    print(f"{algo:<18} | {m['total_sequence_members']:<12} | {m['total_aftershocks']:<12} | {noise_str:<8} | {status_str:<10}")
print("--------------------------------------------------------------------------------")
print('All algorithms reconciled against input candidate scope.')

## 6. Generate Local Smoke Report

Generates `local-smoke-report.json` as specified in JUPYTER_PILOT_WORKFLOW.md Section 5 to authorize Colab migration.

In [ ]:
report_path = exp_out_dir / 'local-smoke-report.json'
smoke_report = generate_local_smoke_report(
    bundle=bundle,
    algorithm_results={
        'WINDOW': res_window,
        'DBSCAN': res_dbscan,
        'HDBSCAN_GLOBAL': res_hdbscan,
    },
    output_path=report_path,
)

assert smoke_report['local_smoke_status'] == 'PASSED'
assert smoke_report['colab_migration_allowed'] is True
print('EXP-02 LOCAL SMOKE REPORT: PASSED')
print('Colab Migration Allowed:', smoke_report['colab_migration_allowed'])
print('Report saved to:', report_path)

# Cleanup temporary directories
if _temp_dir is not None:
    _temp_dir.cleanup()
print('Output artifacts preserved at:', exp_out_dir)
